# **Atelier Prompt Engineering**

### `Contexte`

`Une entreprise souhaite mettre en place « AI Business Assistant », un assistant IA polyvalent capable d'aider ses collaborateurs à exploiter des documents, analyser des données, faire du machine learning et produire des résultats structurés.`

![contexte](images/assistant.png)

## **Partie 1 – Anatomie d'un prompt**

### Objectif
Construire un prompt structuré, à partir de ses composantes, pour répondre au besoin :
« Je souhaite analyser les retours de clients d'une entreprise. »
 
### Composantes utilisées
 
| **`Composante`** | **`Contenu`** |
|---|---|
| **`Rôle`** | Analyste spécialisé en satisfaction client |
| **`Contexte`** | Une entreprise reçoit des retours clients bruts (avis, messages support, réseaux sociaux) et veut en tirer des enseignements exploitables |
| **`Tâche`** | Analyser un lot de retours clients et en extraire les tendances, problèmes récurrents et points positifs |
| **`Données d'entrée`** | Liste de retours clients (texte brut) |
| **`Contraintes`** | Ne pas inventer d'information absente des retours ; rester factuel ; regrouper les retours similaires ; hiérarchiser par fréquence/gravité |
| **`Format de sortie`** | Structuré en sections : résumé général, points positifs, points négatifs, recommandations |
 
### **Prompt final**

```
<role>
Tu es un analyste spécialisé en satisfaction client et retour d'expérience.
</role>

<contexte> 
Une entreprise collecte des retours clients (avis, messages de support, commentaires)
et souhaite en tirer une analyse exploitable pour améliorer ses produits et services.
</context>

<tache>
Analyse la liste de retours clients fournie ci-dessous. Identifie les thèmes récurrents,
distingue les points positifs des points négatifs, et évalue la fréquence ou la gravité
de chaque problème mentionné.
</tache>

<donnees-entree>
{{liste_des_retours_clients}}
</donnees-entree>

<contraines>
- Ne fournis aucune information qui n'apparaît pas explicitement dans les retours fournis.
- Regroupe les retours similaires sous un même thème plutôt que de les lister un par un.
- Reste factuel et objectif, sans jugement de valeur.
- Priorise les problèmes selon leur fréquence apparente.
</contraintes>

<fornat-de-sortie>
1. Résumé général (2-3 phrases)
2. Points positifs identifiés (liste à puces)
3. Points négatifs / problèmes identifiés (liste à puces, du plus au moins fréquent)
4. Recommandations concrètes (liste à puces)
</format-de-sortie>
```

### **Réponse simulée**

```
1. Résumé général
Les retours indiquent une satisfaction sur la qualité du produit et l'ergonomie
de l'interface, mais révèlent des problèmes techniques récurrents (bugs, plantages)
et des lenteurs au niveau du support et de la livraison.

2. Points positifs identifiés
- Qualité du produit jugée satisfaisante
- Interface simple et agréable à utiliser
- Rapport qualité/prix jugé correct malgré un prix perçu comme élevé

3. Points négatifs / problèmes identifiés (du plus au moins fréquent)
- Bugs et plantages de l'application (paiement, connexion après mise à jour, plantages généraux)
- Retards de livraison sans communication proactive
- Lenteur du support client (délai de réponse de plusieurs jours)

4. Recommandations concrètes
- Prioriser la correction des bugs liés au paiement et à la connexion
- Mettre en place des notifications automatiques en cas de retard de livraison
- Réduire le délai de traitement des tickets de support client
```

## **Partie 2 - Comparison des techniques de prompting**

### **Tâche à tester**

Classer : « Le service est rapide mais l'application plante régulièrement. » → classes possibles : positif, négatif, neutre.

### **Zero-shot**
```
Classe le commentaire suivant en positif, négatif ou neutre :
"Le service est rapide mais l'application plante régulièrement."
```
**Réponse: négatif**

### **One-shot**

```
Voici un exemple de classification :
Commentaire : "Le produit est excellent, je le recommande."
Classe : positif

Classe maintenant ce commentaire :
"Le service est rapide mais l'application plante régulièrement."
Classes possibles : positif, négatif, neutre.
```
**Réponse: négatif**

### **Few-shot**
```
Voici des exemples de classification :
Commentaire : "Le produit est excellent, je le recommande." → positif
Commentaire : "Livraison en retard et aucun geste commercial." → négatif
Commentaire : "Le produit correspond à la description, rien de plus." → neutre

Classe ce commentaire selon les mêmes classes (positif, négatif, neutre) :
"Le service est rapide mais l'application plante régulièrement."
```
**Réponse: négatif**

### **Prompt structuré (rôle + contexte + tâche + contraintes + format)**

```
<role>
Tu es un système de classification de sentiment client.
</role>

<contexte>
Les commentaires peuvent contenir à la fois des aspects positifs et négatifs. 
La classe doit refléter le sentiment global dominant.
</contexte>

<tache>
Classe le commentaire suivant.
"Le service est rapide mais l'application plante régulièrement."
</tache>

<contraintes>
- Choisis une seule classe parmi : positif, négatif, neutre.
- Si le commentaire contient à la fois du positif et du négatif, base ta décision sur 
  l'impact perçu le plus fort pour l'utilisateur (ici, un bug récurrent pèse plus 
  qu'une rapidité de service).
- Justifie ta réponse en une phrase.
</contraintes>

<format-de-sortie>
Classe : <positif|négatif|neutre>
Justification : <une phrase>
</format-de-sortie>
```

### **Classe : négatif**

**Justification** : Le bug récurrent de l'application a un impact négatif plus fort sur l'expérience utilisateur que la rapidité du service, qui reste un point secondaire.

## **Partie 3 – Prompt Engineering et raisonnement**

### **3.1 Décomposition du prompt**

**Prompt à décomposer** :

« Analyse ces avis clients et donne-moi les problèmes les plus importants ainsi que les recommandations. »

```
<role>
     Tu es un analyste customer experience.
</role>

<taches>
     <tache-1>
          Lis les avis clients fournis ci-dessous et identifie les problèmes mentionnés.
     </tache-1>

     <tache-2>
          Classe ces problèmes par ordre d'importance, en te basant sur :
          - leur fréquence d'apparition dans les avis
          - leur impact potentiel sur la satisfaction client
     </tache-2>

     <tache-3>
          Tâche (étape 3) :
          Pour chaque problème classé comme important, propose une recommandation concrète et actionnable.
      </tache-3>
</taches>

<donnees>
     {{avis_clients}}
</donnees>

<contraintes>
     - Ne retiens que les problèmes explicitement mentionnés dans les avis.
     - Une recommandation par problème identifié, pas de recommandation générique.
</contraintes>

<format-de-sortie>
     1. Problèmes classés par importance (avec fréquence estimée)
     2. Recommandations associées à chaque problème
</format-de-sortie>
```

#### **Réponse simulé**

![contexte](images/prompt1_3.1.png)

### **3.2 Prompt de vérification / auto-critique**

```
<role>
     Tu es un analyste customer experience.
</role>

<text-a-analyser>
     "L'application plante souvent au moment du paiement (mentionné dans 5 avis sur 8).
     La livraison est jugée lente par 3 clients. 2 clients complimentent le design de 
     l'interface."
</text-a-analyser>

<tache>
     Identifie les problèmes les plus importants et propose des recommandations.
</tache>

<contraintes>
     - Ne t'appuie que sur les faits présents dans le texte.
     - N'invente aucun chiffre ou avis non mentionné.
</contraintes>
```

**Voici la réponse générée précédemment** :
`{{réponse_du_prompt_1}}`


**Tâche** :
Vérifie cette réponse par rapport au texte source original :
`{{texte_source}}`

**Contrôle spécifiquement** :
1. Informations non justifiées : y a-t-il des affirmations qui ne figurent pas dans le texte source ?
2. Contradictions : la réponse se contredit-elle elle-même ou contredit-elle le texte source ?
3. Informations absentes : des éléments importants du texte source ont-ils été omis ?
4. Hallucinations : des chiffres, faits ou avis ont-ils été inventés ?
5. Respect des contraintes : les règles du prompt initial ont-elles été respectées ?

**Format de sortie** :
- Verdict global : conforme / partiellement conforme / non conforme
- Détail par point de contrôle (1 à 5)
- Corrections proposées si nécessaire

### **Réponse simulée (Prompt 2 — vérification) :**

![](images/prompt_verification3.2.png)

## **Partie 4 – Sorties structurées**

### **4.1 Prompt demandant une sortie JSON typée**

**Contexte** : le LLM a produit une réponse en texte libre :
`« Le commentaire semble plutôt négatif. Le client est mécontent du délai de livraison... »`

```
<role>
     Tu es un système d'analyse de sentiment qui produit des sorties structurées 
     pour une application de gestion de la relation client.
</role>

<tache>
Analyse le commentaire client suivant et retourne le résultat exclusivement 
au format JSON.

     <commentaire>
     "Le commentaire semble plutôt négatif. Le client est mécontent du délai de livraison..."
     </commentaire>

     Champs attendus et types :
     - sentiment (string) : sentiment global exprimé
     - categorie (string) : thème principal du commentaire
     - urgence (string) : niveau d'urgence de traitement
     - probleme (string) : description courte du problème identifié
     - confiance (float) : niveau de confiance de l'analyse, entre 0 et 1

     Valeurs autorisées :
     - sentiment : "positif", "negatif", "neutre"
     - urgence : "faible", "moyenne", "élevée"
     - confiance : nombre décimal entre 0.0 et 1.0
</tache>

<contraintes>
     - Retourne uniquement le JSON, sans texte avant ou après.
     - Aucun champ supplémentaire ne doit être ajouté.
     - Si une information n'est pas déterminable avec certitude, réduis la valeur de confiance en conséquence.
</contraintes>
```

![reponse 4.1](images/reponse_4.1.png)

### **4.2 Ajout de la validation**

```
<role>
  Tu es un système d'analyse de sentiment qui produit des sorties structurées 
  strictement validées pour une application de gestion de la relation client.
</role>

<tache>
  Analyse le commentaire client suivant et retourne le résultat au format JSON.

  <commentaire>
  "Le commentaire semble plutôt négatif. Le client est mécontent du délai de livraison..."
  </commentaire>

  Champs attendus et types :
  - sentiment (string)
  - categorie (string)
  - urgence (string)
  - probleme (string)
  - confiance (float)

  Règles de validation strictes (obligatoires) :
  1. Le JSON produit doit être syntaxiquement valide (parsable sans erreur).
  2. Aucune propriété supplémentaire à celles listées ci-dessus n'est autorisée.
  3. Le champ "sentiment" doit valoir uniquement : "positif", "negatif" ou "neutre".
  4. Le champ "confiance" doit être un nombre compris strictement entre 0 et 1 inclus.
  5. Le champ "urgence" doit valoir uniquement : "faible", "moyenne" ou "élevée".
  </taches>

<contraintes>
  - Ne retourne que le JSON, sans commentaire, balise markdown ou texte additionnel.
  - Si une règle ne peut pas être respectée avec les informations disponibles, 
    indique le champ concerné avec la valeur la plus prudente possible plutôt 
    que d'inventer une donnée.
</contraintes>
```

![reponse 4.2](images/reponse4.2.png)

## **Partie 5 – Prompts pour les applications métier**

### **5.1 — Résumé de document**

```
<role>
     Tu es un assistant de synthèse documentaire pour un usage professionnel.
</role>

<tache>
     Résume le document fourni ci-dessous.
</tache>

Document :
{{document}}

<contraintes>
     - Maximum 250 mots.
     - Conserve toutes les informations factuelles importantes (chiffres, dates, noms).
     - Identifie clairement les objectifs mentionnés dans le document.
     - Identifie clairement les résultats obtenus.
     - Identifie clairement les recommandations formulées.
     - N'invente aucune information absente du document.
</contraintes>

<format-de-sortie>
     - Objectifs :
     - Résultats :
     - Recommandations :
     - Résumé synthétique (max 250 mots au total)
</format-de-sortie>
```

![reponse 5.1](images/reponse5.1.png)

### **5.2 — Traduction Fançais → Englais**

```
<role>
     Tu es un traducteur technique professionnel français → anglais.
</role>

<taches>
     Traduis le document suivant du français vers l'anglais.

     Document :
     {{document}}
</taches>
<contraintes>
     - Conserve fidèlement le sens du texte original.
     - Conserve la structure du document (paragraphes, listes, titres).
     - Conserve les termes techniques (ne les traduis pas de façon approximative 
     ou par un équivalent générique s'ils ont un sens précis dans le domaine).
     - Ne résume pas le contenu : traduction intégrale.
     - N'ajoute aucune information, explication ou commentaire absent de l'original.
</contraintes>
<format-de-sortie>
     Texte traduit uniquement, sans balise ni commentaire additionnel.
</format-de-sortie>
```

![reponse 5.2](images/reponse5.2.png)

### **5.3 — Classification de ticket informatique**

```
<role>
Tu es un système de triage de tickets support informatique.
</role>

<tache>
     Classe le ticket suivant dans une des catégories : réseau, logiciel, matériel, 
     sécurité, accès, autre.

     <ticket>
          {{contenu_ticket}}
     </ticket>
</tache>

<contraintes>
     - Une seule catégorie doit être choisie.
     - Si plusieurs catégories semblent pertinentes, choisis celle correspondant à la cause racine la plus probable.
</contraintes>

<format-de-sortie>
{
  "categorie": "<réseau|logiciel|matériel|sécurité|accès|autre>",
  "justification": "<une phrase expliquant le choix>"
}
</format-de-sortie>
```

![reponse 5.3](images/reponse5.3.png)

## **5.4 — Extraction d'informations depuis une facture**

```
<role>
     Tu es un système d'extraction de données comptables.
</role>

<taches>
     Extrait les informations suivantes de la facture fournie : numéro_facture, 
     date, client, montant_ht, tva, montant_ttc.

     <factures>
          {{texte_facture}}
     </factures>
</taches>

<contrainte>
     - Retourne uniquement un JSON valide.
     - Utilise la valeur null pour toute information absente ou non identifiable, 
     ne l'invente jamais.
     - Les montants doivent être des nombres (pas de texte, pas de symbole monétaire).
</contrainte>
</format-de-sortie>
     {
     "numero_facture": "",
     "date": "",
     "client": "",
     "montant_ht": 0,
     "tva": 0,
     "montant_ttc": 0
     }
</format-de-sortie>
```

![reponse 5.4](images/reponse5.4.png)

## **5.5 — Email de retard de livraison**

```
<role>
     Tu es un chargé de relation client rédigeant un email professionnel.
</role>
<tache>
  Rédige un email destiné à un client dont la livraison a pris du retard.

  <objectifs>
    - Reconnaître clairement le retard.
    - Présenter des excuses.
    - Expliquer la situation sans inventer de cause si elle n'est pas connue.
    - Proposer une solution ou un geste commercial.
  </objectifs>
  <ton>
    Professionnel, courtois et rassurant.
  </ton>
</tache>

<contraintes>
  - Longueur maximale : 150 mots.
  - N'invente aucune cause précise du retard si elle n'est pas fournie ; reste générique si nécessaire ("un contretemps logistique").
</contraintes>
```

![](images/reponse5.5.png)

## **Partie 6 – Prompt Engineering pour le Machine Learning**

### **6.1 — Stratégies de traitement des données (valeurs manquantes, doublons, outliers, catégorielles)**

```
<role>
     Tu es un data scientist spécialisé en préparation de données pour l'IoT.
</role>

<contexte>
     On dispose d'un dataset de capteurs (température, humidité, consommation 
     énergétique, timestamp, type de capteur) issu d'un bâtiment instrumenté.
</contexte>

<taches>
     Propose des stratégies de traitement pour chacun des problèmes suivants 
     présents dans ce dataset : valeurs manquantes, doublons, valeurs aberrantes, 
     variables catégorielles.
</taches>

<contraintes>
     - Pour chaque problème, indique : la méthode de détection, la méthode de 
     traitement recommandée, et les risques associés à ce traitement.
     - Adapte les recommandations au contexte de séries temporelles issues de capteurs.
</contraintes>

<format-de-sortie>
     Un tableau avec les colonnes : Problème | Détection | Traitement | Risques
</format-de-sortie>
```

![](images/reponse6.1.png)

### **6.2 — Visualisations pertinentes pour la consommation énergétique**

```
<role>
     Tu es un data analyst spécialisé en efficacité énergétique des bâtiments.
</role>
<tache>
     À partir du dataset de capteurs (température, humidité, consommation 
     énergétique, timestamp, zone), propose les visualisations les plus 
     pertinentes pour comprendre la consommation énergétique du bâtiment.
</tache>
<contraintes>
     Pour chaque visualisation, précise : le type de graphique, les variables 
     utilisées, l'objectif, et l'interprétation attendue.
</contraintes>
<format-de-sortie>
     Liste numérotée avec les 4 éléments pour chaque visualisation.
</format-de-sortie>
```

![](images/reponse6.2.png)

### **6.3 — Modèles adaptés à la prédiction de consommation énergétique**

```
<role>
  Tu es un data scientist spécialisé en prédiction de séries temporelles énergétiques.
</role>
<tache>
  À partir du dataset de capteurs, propose plusieurs modèles adaptés à la 
  prédiction de la consommation énergétique d'un bâtiment.
</tache>
<contraintes>
  Pour chaque modèle, indique : le principe, les avantages, les limites, 
  le type de problème (régression/classification/série temporelle), et 
  les métriques pertinentes.
</contraintes>
<format-de-sortie>
  Tableau avec les colonnes : Modèle | Principe | Avantages | Limites | Type | Métriques
</format-de-sortie>
```

### **Réponse simulée**
### Modèles adaptés à la prédiction de consommation énergétique

| Modèle | Principe | Avantages | Limites | Type | Métriques |
|---|---|---|---|---|---|
| **Régression linéaire** | Relation linéaire entre variables (température, heure...) et consommation | Simple, interprétable, rapide | Ne capture pas les relations non-linéaires ni la saisonnalité complexe | Régression | MAE, RMSE |
| **Random Forest Regressor** | Ensemble d'arbres de décision | Capture les non-linéarités, robuste aux outliers | Moins interprétable, coûteux en mémoire sur gros volumes | Régression | MAE, RMSE, R² |
| **SARIMA** | Modèle statistique de série temporelle avec saisonnalité | Bien adapté aux cycles journaliers/hebdomadaires réguliers | Nécessite une série stationnaire, moins efficace avec variables exogènes multiples | Série temporelle | MAE, RMSE, MAPE |
| **LSTM** (réseau de neurones récurrent) | Apprentissage de dépendances temporelles longues | Excellent pour capter des motifs complexes et non-linéaires | Nécessite beaucoup de données, coût de calcul élevé, moins interprétable | Série temporelle | MAE, RMSE, MAPE |

### **6.4 — Métriques de classification**

```
<role>
     Tu es un formateur en machine learning avec 10 ans d'experience
</role>
<taches>
     Explique les métriques de classification suivantes : Accuracy, Precision, 
     Recall, F1-score, ROC-AUC.
</taches>
<contraintes>
     Pour chacune, donne : la définition, l'interprétation, un exemple concret, 
     et le contexte où elle est particulièrement utile.
</contraintes>
```

Accuracy
- Définition : proportion de prédictions correctes sur l'ensemble des prédictions.
- Interprétation : plus elle est proche de 1, meilleure est la performance globale.
- Exemple : sur 100 tickets classés, 90 bien classés → accuracy = 0.90.
- Contexte utile : classes équilibrées (ex. autant de ticket "logiciel" que "matériel").

Precision
- Définition : parmi les prédictions positives, proportion réellement correctes.
- Interprétation : mesure le taux de faux positifs.
- Exemple : sur 20 alertes "panne capteur" prédites, 15 sont réelles → precision = 0.75.
- Contexte utile : quand un faux positif coûte cher (ex. alerte inutile envoyée à un technicien).

Recall
- Définition : parmi les cas réellement positifs, proportion détectée par le modèle.
- Interprétation : mesure le taux de faux négatifs.
- Exemple : sur 20 pannes réelles, seulement 12 détectées → recall = 0.60.
- Contexte utile : quand rater un cas positif est critique (ex. détection de panne 
  électrique dangereuse).

F1-score
- Définition : moyenne harmonique entre precision et recall.
- Interprétation : équilibre les deux métriques, utile quand les classes sont déséquilibrées.
- Exemple : precision = 0.75, recall = 0.60 → F1 ≈ 0.67.
- Contexte utile : dataset déséquilibré où accuracy seule serait trompeuse.

ROC-AUC
- Définition : aire sous la courbe ROC (taux de vrais positifs vs taux de faux positifs).
- Interprétation : proche de 1 = excellente séparation des classes ; 0.5 = aléatoire.
- Exemple : AUC = 0.92 → le modèle distingue très bien pannes/non-pannes.
- Contexte utile : comparer plusieurs modèles indépendamment du seuil de décision choisi.

## **6.5 — Métriques de régression**

```
<role>
     Tu es un formateur en machine learning.
</role>
<taches>
     Explique les métriques de régression suivantes : MAE, MSE, RMSE.
</taches>
<contraintes>
     Pour chacune, donne : la définition, l'interprétation, un exemple concret, 
     et le contexte où elle est particulièrement utile.
</contraintes>
```

![](images/reponse6.5.png)

## **Partie 7 — Prompt Engineering et RAG.**

### **7.1 Document utilisé pour le test**

![](images/reponse7.1.png)

### **Prompt A — sans document**

```
Réponds à la question suivante :
"Quelle est l'autonomie de la batterie du capteur EnviroSense X200 et 
peut-il être immergé dans l'eau ?"
```

![](images/reponse7.2_a.png)

## **Prompt B — avec document, sans contrainte**

```
Voici un document :
"[Fiche technique — Capteur EnviroSense X200 ... texte complet ci-dessus]"

Réponds à la question suivante :
"Quelle est l'autonomie de la batterie du capteur EnviroSense X200 et 
peut-il être immergé dans l'eau ?"
```

![](images/reponse7.2_b.png)

### **Prompt C — avec document + contraintes anti-hallucination**

```
Voici un document de référence :
"[Fiche technique — Capteur EnviroSense X200 ... texte complet ci-dessus]"

Réponds à la question suivante en utilisant UNIQUEMENT les informations 
contenues dans le document ci-dessus :
<question>
     "Quelle est l'autonomie de la batterie du capteur EnviroSense X200 et 
     peut-il être immergé dans l'eau ?"
</question>
<contraintes>
     - N'utilise aucune connaissance externe au document fourni.
     - Si une partie de la question n'a pas de réponse dans le document, indique-le explicitement.
     - Cite le passage exact du document sur lequel tu bases chaque affirmation.
</contraintes>
```

![](images/reponse7.2_c.png)

## **7.3 Comparaison des résultats (Prompt A vs B vs C)**

| Critère | Prompt A (sans doc) | Prompt B (avec doc) | Prompt C (avec doc + contraintes) |
|---|---|---|---|
| **Exactitude** | ❌ Fausse (invention "6 mois à 2 ans") | ✅ Correcte | ✅ Correcte |
| **Traçabilité** | Aucune | Faible (pas de citation) | ✅ Forte (citations explicites) |
| **Risque d'hallucination** | Élevé | Faible | Très faible |
| **Fiabilité pour usage métier** | Non exploitable | Acceptable | Optimale |

**Observation** : sans contexte documentaire, le LLM comble le vide par des 
connaissances générales potentiellement fausses. Fournir le document résout 
déjà la majorité des hallucinations, mais ajouter des contraintes explicites 
de traçabilité et de non-invention rend la réponse réellement fiable et 
vérifiable pour un usage professionnel.

## **Partie 8 — Évaluation et optimisation des prompts.**


**Texte à résumer (utilisé pour les 3 versions)**
```
Texte source :
"Le marché des capteurs IoT pour bâtiments intelligents connaît une croissance soutenue, portée par la demande en efficacité énergétique et en conformité réglementaire. En 2025, le secteur a enregistré une hausse de 22% des 
déploiements par rapport à l'année précédente, principalement dans les secteurs tertiaire et industriel. Les principaux freins identifiés restent le coût d'installation initial, la complexité d'intégration avec les 
systèmes existants, et les préoccupations liées à la cybersécurité des objets connectés. Les acteurs du marché misent désormais sur des solutions plug-and-play et des standards de communication ouverts pour lever ces 
obstacles. Les prévisions tablent sur une croissance annuelle de 18% jusqu'en 2028."
```